In [ ]:
import jax
import jax.numpy as jnp
import equinox as eqx

from llm_equilibrium_core import TripletModel, Dataset, TrainConfig, train, get_S, load_csv

jax.config.update("jax_enable_x64", True)


class MLP_DER(TripletModel):
    # Explicitly define the layers so Equinox registers them correctly
    K: jax.Array
    line1: eqx.nn.Linear
    line2: eqx.nn.Linear
    line3: eqx.nn.Linear

    def __init__(self, K0: jax.Array, key: jax.Array):
        self.K = K0
        in_size = K0.size
        width = 16
        out_size = 1

        k1, k2, k3 = jax.random.split(key, 3)

        self.line1 = eqx.nn.Linear(in_size, width, key=k1)
        self.line2 = eqx.nn.Linear(width, width, key=k2)
        self.line3 = eqx.nn.Linear(width, out_size, key=k3)

    def __call__(self, del_strain: jax.Array) -> jax.Array:
        # Standard forward pass using Equinox layers
        x = self.line1(del_strain)
        x = jax.nn.relu(x)
        x = self.line2(x)
        x = jax.nn.relu(x)
        return jnp.sum(self.K * del_strain**2) + jax.nn.softplus(x)[0]


dataset_train = Dataset.from_npz("dlo_DLO_slinky_2d_rect_155pts_8428cea6.csv.npz")
dataset_valid = Dataset.from_npz("dlo_DLO_slinky_2d_rect_1836pts_83082e60.csv.npz")
config = TrainConfig(epochs=1000, lr=1e-3, S_factor=0.0)
model, train_loss, valid_loss = train(
    MLP_DER, dataset_train, dataset_valid, config=config
)

Starting training for 1000 epochs...
Epoch 0100 | Train Loss: 1.141930 | Valid Loss: 1.231888
Epoch 0200 | Train Loss: 0.057392 | Valid Loss: 0.072741
Epoch 0300 | Train Loss: 0.039025 | Valid Loss: 0.060187
Epoch 0400 | Train Loss: 0.026031 | Valid Loss: 0.053804
Epoch 0500 | Train Loss: 0.018633 | Valid Loss: 0.048467
Epoch 0600 | Train Loss: 0.013776 | Valid Loss: 0.048394
Epoch 0700 | Train Loss: 0.016320 | Valid Loss: 0.048264
Epoch 0800 | Train Loss: 0.019851 | Valid Loss: 0.056772
Epoch 0900 | Train Loss: 0.020591 | Valid Loss: 0.051014
Epoch 1000 | Train Loss: 0.019163 | Valid Loss: 0.045980


In [ ]:
import os

for file in os.listdir("data"):
    trajectories = load_csv(f"data/{file}", is_2d=True)
    qs = trajectories.reshape(trajectories.shape[0], -1)
    S = get_S(trajectories)
    dataset = Dataset(qs=jnp.asarray(qs), S=jnp.asarray(S))
    dataset.to_npz(f"{file}.npz")